<a href="https://colab.research.google.com/github/Betty-jyw/MMA3001-Occupancy-Prediction/blob/main/notebooks/05_final_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MMA3001 — Final Results and Discussion

## 30-Minute-Ahead Room Occupancy Prediction

### Project Overview

This project investigates the use of historical occupancy sensor data to predict whether a selected room will be occupied 30 minutes into the future.

The analysis follows five main stages:

1. Data exploration and preprocessing.
2. Time-based feature engineering.
3. Machine learning model development.
4. Model evaluation and error analysis.
5. Final interpretation, limitations, and conclusions.

### Research Objective

The primary objective is to evaluate whether machine learning models can improve 30-minute-ahead room occupancy prediction compared with a persistence baseline.

Three approaches are considered:

- Persistence Baseline
- Logistic Regression
- Random Forest

### Scope of This Notebook

This notebook consolidates the final experimental results, interprets the main findings, discusses limitations, and identifies potential improvements.

The analysis uses results obtained from the previous notebooks. No additional model training is required.

## 1. Load Final Experimental Results

This section loads the final predictions generated during model development and evaluation.

The results are used to summarise the performance of the Persistence Baseline, Logistic Regression, and Random Forest models on the same chronological test dataset.

All metrics are calculated from the saved predictions to ensure consistency with the previous notebooks.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from google.colab import drive

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Connect Google Drive
drive.mount("/content/drive")

# Define project directories
project_dir = Path("/content/drive/MyDrive/MMA3001_Project")
results_dir = project_dir / "results"

# Load final predictions
predictions_path = results_dir / "final_model_predictions.csv"

predictions_df = pd.read_csv(
    predictions_path,
    index_col=0,
    parse_dates=True
)

predictions_df = predictions_df.sort_index()

# Check data integrity
assert len(predictions_df) > 0
assert not predictions_df.isna().any().any()

expected_columns = [
    "actual",
    "baseline_pred",
    "logistic_pred",
    "random_forest_pred"
]

assert all(
    col in predictions_df.columns
    for col in expected_columns
)

print("Final experimental results loaded successfully.")
print("Dataset shape:", predictions_df.shape)

print("\nTest period:")
print("Start:", predictions_df.index.min())
print("End:", predictions_df.index.max())

print("\nFirst five rows:")
display(predictions_df.head())

Mounted at /content/drive
Final experimental results loaded successfully.
Dataset shape: (21412, 4)

Test period:
Start: 2025-09-12 06:05:00+00:00
End: 2026-04-22 12:30:00+00:00

First five rows:


,actual,baseline_pred,logistic_pred,random_forest_pred
collecteddate,,,,
2025-09-12 06:05:00+00:00,1,1,1,1
2025-09-12 06:10:00+00:00,1,1,1,1
2025-09-12 06:15:00+00:00,1,1,1,1
2025-09-12 06:20:00+00:00,1,0,1,1
2025-09-12 06:25:00+00:00,1,1,1,1


## 2. Final Model Performance Comparison

This section summarises the predictive performance of the three models on the chronological test dataset.

Four classification metrics are used:

- **Accuracy:** Proportion of all predictions that are correct.
- **Precision:** Proportion of predicted occupied states that are actually occupied.
- **Recall:** Proportion of actual occupied states correctly identified.
- **F1-score:** Harmonic mean of precision and recall.

The results provide a consistent comparison between the Persistence Baseline, Logistic Regression, and Random Forest models.

In [2]:
# Calculate final classification metrics
model_columns = {
    "Persistence Baseline": "baseline_pred",
    "Logistic Regression": "logistic_pred",
    "Random Forest": "random_forest_pred"
}

y_true = predictions_df["actual"]

performance_results = []

for model_name, prediction_column in model_columns.items():
    y_pred = predictions_df[prediction_column]

    performance_results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0)
    })

performance_df = pd.DataFrame(performance_results)
performance_df = performance_df.set_index("Model")

print("Final Model Performance Summary")
display(performance_df.round(4))

# Save results for the final report
performance_path = results_dir / "final_performance_summary.csv"

performance_df.to_csv(performance_path)

print("\nPerformance summary saved to:")
print(performance_path)

Final Model Performance Summary


,Accuracy,Precision,Recall,F1-score
Model,,,,
Persistence Baseline,0.6740,0.7511,0.7634,0.7572
Logistic Regression,0.7168,0.7430,0.8788,0.8052
Random Forest,0.6898,0.7359,0.8332,0.7815



Performance summary saved to:
/content/drive/MyDrive/MMA3001_Project/results/final_performance_summary.csv


## 3. Interpretation of Final Model Performance

### 3.1 Overall Model Comparison

The Logistic Regression model achieved the best overall performance on the chronological test dataset, with an accuracy of 71.68%, recall of 87.88%, and F1-score of 0.8052.

Compared with the Persistence Baseline, Logistic Regression improved accuracy by 4.28 percentage points and F1-score by 0.0480.

Random Forest achieved an accuracy of 68.98% and an F1-score of 0.7815. Although it outperformed the Persistence Baseline, it did not achieve the same performance as Logistic Regression.

### 3.2 Precision and Recall Trade-off

The Persistence Baseline achieved the highest precision (0.7511), while Logistic Regression achieved the highest recall (0.8788).

This indicates that Logistic Regression was more effective at identifying future occupied states, but it also produced more false positive predictions.

Therefore, the choice of model depends on the intended application. High recall may be beneficial when missing an occupied room is costly, whereas false positives may lead to unnecessary building energy consumption.

### 3.3 Model Selection

Logistic Regression was selected as the preferred model because it achieved the highest accuracy and F1-score among the three evaluated approaches.

However, its relatively high false positive rate and tendency to overpredict occupancy should be considered before practical deployment.

The results demonstrate that a relatively simple machine learning model can outperform both the Persistence Baseline and the more complex Random Forest model for this particular prediction task.

## 4. Classification Error Analysis

This section examines the types of classification errors made by each model.

Two error categories are considered:

- **False Positive (FP):** The model predicts that a room will be occupied when it is actually unoccupied.
- **False Negative (FN):** The model predicts that a room will be unoccupied when it is actually occupied.

False positives may result in unnecessary energy consumption if predictions are used for building control.

False negatives may affect occupant comfort if building systems incorrectly assume that a room is unoccupied.

Comparing these errors provides additional insight beyond overall accuracy and F1-score.

In [3]:
from sklearn.metrics import confusion_matrix

# Calculate classification errors for each model
error_results = []

for model_name, prediction_column in model_columns.items():

    y_pred = predictions_df[prediction_column]

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    error_results.append({
        "Model": model_name,
        "False Positives": fp,
        "False Negatives": fn,
        "Total Errors": fp + fn,
        "False Positive Rate": fp / (fp + tn),
        "False Negative Rate": fn / (fn + tp)
    })

final_error_df = pd.DataFrame(error_results)
final_error_df = final_error_df.set_index("Model")

print("Final Classification Error Summary")
display(final_error_df.round(4))

# Save results
error_summary_path = results_dir / "final_error_summary.csv"

final_error_df.to_csv(error_summary_path)

print("\nError summary saved to:")
print(error_summary_path)

Final Classification Error Summary


,False Positives,False Negatives,Total Errors,False Positive Rate,False Negative Rate
Model,,,,,
Persistence Baseline,3607,3374,6981,0.5044,0.2366
Logistic Regression,4335,1728,6063,0.6062,0.1212
Random Forest,4264,2379,6643,0.5963,0.1668



Error summary saved to:
/content/drive/MyDrive/MMA3001_Project/results/final_error_summary.csv


### 4.1 Comparison of Classification Errors

The Logistic Regression model produced the lowest total number of classification errors (6,063), compared with 6,981 for the Persistence Baseline and 6,643 for Random Forest.

Logistic Regression also achieved the lowest number of false negatives (1,728), representing a reduction of approximately 48.8% compared with the Persistence Baseline.

However, Logistic Regression generated 4,335 false positives, which was higher than both the Persistence Baseline (3,607) and Random Forest (4,264).

These findings indicate that Logistic Regression was more effective at identifying occupied states but less effective at correctly identifying unoccupied states.

### 4.2 False Positive and False Negative Rates

The False Positive Rate (FPR) measures the proportion of actual unoccupied states incorrectly classified as occupied.

Logistic Regression had the highest FPR (60.62%), followed by Random Forest (59.63%) and the Persistence Baseline (50.44%).

In contrast, Logistic Regression achieved the lowest False Negative Rate (FNR) of 12.12%, compared with 16.68% for Random Forest and 23.66% for the Persistence Baseline.

This demonstrates a trade-off between detecting occupied states and avoiding unnecessary occupied predictions.

### 4.3 Implications for Building Management

In practical building management applications, different classification errors may have different consequences.

False positives may lead to unnecessary operation of heating, cooling, or lighting systems when a room is unoccupied.

False negatives may cause building systems to reduce services while occupants are present, potentially affecting comfort.

Therefore, although Logistic Regression achieved the best overall predictive performance, its high false positive rate may limit its usefulness for energy-saving applications without further improvements.

Future work could investigate alternative decision thresholds, cost-sensitive learning, and additional sensor features to achieve a more suitable balance between the two error types.


## 5. Occupancy Prediction Bias Analysis

This section investigates whether the models systematically overestimate or underestimate room occupancy.

The actual occupancy rate is calculated as the proportion of occupied observations in the test dataset.

For each model, the predicted occupancy rate is calculated as the proportion of predictions classified as occupied.

Prediction bias is defined as:

**Prediction Bias = Predicted Occupancy Rate − Actual Occupancy Rate**

A positive bias indicates that the model overpredicts occupancy, while a negative bias indicates underprediction.

This analysis helps assess whether the models may produce unnecessarily frequent occupied predictions.


In [4]:

# Calculate actual occupancy rate
actual_occupancy_rate = predictions_df["actual"].mean()

# Calculate predicted occupancy rates and bias
bias_results = []

for model_name, prediction_column in model_columns.items():

    predicted_rate = predictions_df[prediction_column].mean()

    prediction_bias = predicted_rate - actual_occupancy_rate

    bias_results.append({
        "Model": model_name,
        "Actual Occupancy Rate": actual_occupancy_rate,
        "Predicted Occupancy Rate": predicted_rate,
        "Prediction Bias": prediction_bias,
        "Bias (percentage points)": prediction_bias * 100
    })

final_bias_df = pd.DataFrame(bias_results)
final_bias_df = final_bias_df.set_index("Model")

print("Final Occupancy Prediction Bias Summary")
display(final_bias_df.round(4))

# Save results
bias_summary_path = results_dir / "final_occupancy_bias_summary.csv"

final_bias_df.to_csv(bias_summary_path)

print("\nOccupancy bias summary saved to:")
print(bias_summary_path)


Final Occupancy Prediction Bias Summary


,Actual Occupancy Rate,Predicted Occupancy Rate,Prediction Bias,Bias (percentage points)
Model,,,,
Persistence Baseline,0.666,0.6769,0.0109,1.0882
Logistic Regression,0.666,0.7878,0.1218,12.1754
Random Forest,0.666,0.7541,0.0880,8.8035



Occupancy bias summary saved to:
/content/drive/MyDrive/MMA3001_Project/results/final_occupancy_bias_summary.csv



### 5.1 Comparison of Occupancy Prediction Bias

The actual occupancy rate in the test dataset was 66.60%.

The Persistence Baseline predicted an occupancy rate of 67.69%, resulting in a positive bias of 1.09 percentage points.

Logistic Regression predicted an occupancy rate of 78.78%, producing the largest positive bias of 12.18 percentage points.

Random Forest predicted an occupancy rate of 75.41%, corresponding to a positive bias of 8.80 percentage points.

All three models therefore overestimated the overall proportion of occupied states, although the magnitude of the bias varied considerably.

### 5.2 Relationship Between Prediction Bias and Classification Errors

The positive occupancy prediction bias observed in Logistic Regression is consistent with its relatively high false positive rate of 60.62%.

Although Logistic Regression achieved the highest F1-score (0.8052), it tended to classify unoccupied observations as occupied.

In comparison, the Persistence Baseline produced the smallest occupancy rate bias, despite having the lowest overall classification accuracy.

This demonstrates that a model with better classification performance does not necessarily produce a more accurate estimate of the overall occupancy proportion.

### 5.3 Practical Implications

Overprediction of occupancy may reduce the potential energy savings of occupancy-based building control systems.

For example, if predicted occupancy were used to determine whether heating, cooling, or lighting should remain active, false occupied predictions could result in unnecessary operation.

However, the current study did not measure actual building energy consumption. Therefore, these implications remain hypothetical rather than experimentally verified.

Future research could investigate probability calibration, alternative classification thresholds, and cost-sensitive evaluation to reduce occupancy prediction bias while maintaining acceptable recall.



## 6. Temporal Performance Analysis

This section evaluates how model performance changes across different months of the test period.

Monthly accuracy and F1-score are calculated for the Persistence Baseline, Logistic Regression, and Random Forest models.

Timestamps are converted from UTC to Australia/Melbourne local time before monthly grouping.

This analysis helps identify temporal variations in predictive performance and assess whether the selected model performs consistently throughout the test period.


In [5]:

# Prepare predictions for monthly evaluation
monthly_predictions = predictions_df.copy()

# Convert timestamps to Melbourne local time
monthly_predictions.index = (
    monthly_predictions.index.tz_convert("Australia/Melbourne")
)

# Create monthly grouping labels
monthly_predictions["month"] = (
    monthly_predictions.index.strftime("%Y-%m")
)

# Calculate monthly model performance
monthly_results = []

for month, month_data in monthly_predictions.groupby("month"):

    actual = month_data["actual"]

    for model_name, prediction_column in model_columns.items():

        predicted = month_data[prediction_column]

        monthly_results.append({
            "Month": month,
            "Model": model_name,
            "Samples": len(month_data),
            "Accuracy": accuracy_score(actual, predicted),
            "F1-score": f1_score(
                actual,
                predicted,
                zero_division=0
            )
        })

final_monthly_df = pd.DataFrame(monthly_results)

print("Monthly Model Performance")
display(final_monthly_df.round(4))

# Save results
monthly_summary_path = (
    results_dir / "final_monthly_performance_summary.csv"
)

final_monthly_df.to_csv(
    monthly_summary_path,
    index=False
)

print("\nMonthly performance summary saved to:")
print(monthly_summary_path)


Monthly Model Performance


,Month,Model,Samples,Accuracy,F1-score
0,2025-09,Persistence Baseline,1897,0.6927,0.7802
1,2025-09,Logistic Regression,1897,0.7364,0.8257
2,2025-09,Random Forest,1897,0.7180,0.8093
3,2025-10,Persistence Baseline,4044,0.6761,0.7698
4,2025-10,Logistic Regression,4044,0.7295,0.8227
5,2025-10,Random Forest,4044,0.6963,0.7931
6,2025-11,Persistence Baseline,3295,0.6455,0.6887
7,2025-11,Logistic Regression,3295,0.6753,0.7375
8,2025-11,Random Forest,3295,0.6580,0.7203
9,2025-12,Persistence Baseline,2077,0.6052,0.6563



Monthly performance summary saved to:
/content/drive/MyDrive/MMA3001_Project/results/final_monthly_performance_summary.csv



### 6.1 Monthly Performance Comparison

The monthly evaluation covered eight months, from September 2025 to April 2026.

Logistic Regression consistently achieved the highest accuracy and F1-score among the three evaluated models in every month.

Its monthly accuracy ranged from 63.70% in December 2025 to 78.02% in March 2026.

Similarly, its F1-score ranged from 0.7155 in December 2025 to 0.8625 in March 2026.

Although Logistic Regression consistently outperformed the other models, its absolute predictive performance varied across the test period.

### 6.2 Temporal Variation in Prediction Performance

All three models experienced changes in performance across different months.

For Logistic Regression, performance declined during November and December 2025 before improving in the following months.

The strongest performance was observed in March 2026, when Logistic Regression achieved an accuracy of 78.02% and an F1-score of 0.8625.

These results suggest that the predictability of room occupancy may vary over time.

Changes in occupancy patterns, room usage, or the distribution of occupied and unoccupied states may contribute to these differences. However, the present analysis does not establish their specific causes.

### 6.3 Implications for Model Reliability

The results indicate that Logistic Regression maintained its relative advantage over the Persistence Baseline and Random Forest throughout the evaluation period.

However, the variation in monthly accuracy and F1-score suggests that its performance should not be assumed to remain constant over time.

For practical deployment, regular performance monitoring may be necessary to identify changes in prediction quality.

Future work could investigate periodic model retraining, additional temporal features, and evaluation across multiple rooms to improve reliability under changing occupancy conditions.



## 7. Project Limitations

Although the results demonstrate that machine learning can improve 30-minute-ahead occupancy prediction, several limitations should be considered.

### 7.1 Single-Room Dataset

The analysis focused on one selected room rather than multiple rooms across the building.

As a result, the findings reflect the occupancy patterns of this particular space and may not generalise to rooms with different functions, usage schedules, or occupancy behaviours.

Further evaluation across multiple rooms would be necessary to assess the generalisability of the models.

### 7.2 Limited Sensor Features

The prediction models primarily relied on historical occupancy states and time-related features.

Although environmental sensor data were available, the environmental sensors could not be reliably matched to the selected occupancy space using the available location information.

Therefore, variables such as CO2 concentration, temperature, humidity, and light intensity were not included in the final models.

The absence of these features may limit the ability of the models to identify changes in occupancy patterns.

### 7.3 High False Positive Rate

The selected Logistic Regression model achieved the highest overall F1-score but also produced a relatively high false positive rate of 60.62%.

This indicates that many actually unoccupied observations were incorrectly classified as occupied.

Such errors may reduce the potential benefits of occupancy prediction in energy-efficient building management applications.

### 7.4 Temporal Variation

Monthly evaluation revealed considerable variation in model performance.

For Logistic Regression, accuracy ranged from 63.70% to 78.02% across the evaluation period.

This suggests that predictive performance may change over time, potentially due to changes in occupancy patterns or room usage.

However, the available results do not establish the underlying causes of these variations.

### 7.5 Retrospective Evaluation

The models were evaluated using historical sensor observations and a chronological train-test split.

Although this approach preserves the temporal ordering of the data, it does not fully represent the challenges of real-time deployment.

The study did not evaluate live sensor delays, operational reliability, or integration with building control systems.

Consequently, the reported performance should be interpreted as retrospective predictive performance rather than demonstrated real-world operational effectiveness.



## 8. Future Improvements

Based on the experimental findings and identified limitations, several directions could be explored to improve the accuracy, reliability, and practical usefulness of future occupancy prediction models.

### 8.1 Incorporating Environmental Sensor Data

Future research could investigate the integration of environmental variables, including CO2 concentration, temperature, humidity, and light intensity.

These variables may provide additional information about room usage and environmental changes associated with occupancy.

However, reliable spatial matching between occupancy sensors and environmental sensors would be required before combining these datasets.

### 8.2 Improving Classification Thresholds

The Logistic Regression model achieved the highest F1-score but also produced a relatively high false positive rate.

Future work could investigate alternative classification thresholds to achieve a more suitable balance between precision and recall.

Threshold selection should be performed using a separate validation dataset rather than the final test dataset.

Cost-sensitive evaluation could also be considered if the consequences of false positives and false negatives can be quantified.

### 8.3 Expanding the Dataset to Multiple Rooms

The current study focused on one selected room.

Future research could evaluate the same prediction methods across multiple rooms with different usage patterns.

This would help determine whether the observed performance improvements generalise beyond the selected space.

It could also support the development of models that account for differences between room types and occupancy behaviours.

### 8.4 Improving Temporal Features

The current models used historical occupancy states, lagged occupancy features, recent occupancy rates, and time-related variables.

Future work could investigate additional temporal features, such as longer historical windows, day-type indicators, and recurring occupancy patterns.

These features may help capture changes in room usage that are not fully represented by the current feature set.

Any additional features should be constructed using information available at the prediction time to avoid data leakage.

### 8.5 Periodic Model Evaluation and Retraining

The monthly evaluation demonstrated that prediction performance varied throughout the test period.

Future work could investigate periodic model evaluation and retraining using newly collected observations.

This may help models adapt to changing occupancy patterns.

However, retraining strategies should be evaluated using appropriate chronological validation to ensure that improvements generalise to future observations.

### 8.6 Real-World Deployment and Energy Evaluation

The current project evaluated predictive performance using historical data.

A future study could investigate the deployment of the selected model in a real-time building monitoring system.

Such an evaluation could consider prediction latency, sensor reliability, occupant comfort, and building energy consumption.

This would help determine whether improvements in occupancy prediction translate into measurable operational benefits.



## 9. Final Conclusion

This project investigated the prediction of room occupancy 30 minutes into the future using historical occupancy sensor data and machine learning techniques.

The study focused on one selected room and used five-minute occupancy observations to construct lagged occupancy features, recent occupancy rates, and time-related variables.

Three prediction approaches were evaluated: a Persistence Baseline, Logistic Regression, and Random Forest. Their performance was compared using a chronological train-test split to preserve the temporal ordering of the observations.

### 9.1 Main Findings

Logistic Regression achieved the best overall predictive performance, with an accuracy of 71.68%, recall of 87.88%, and F1-score of 0.8052.

Compared with the Persistence Baseline, Logistic Regression improved accuracy by 4.28 percentage points and increased the F1-score from 0.7572 to 0.8052.

Random Forest also outperformed the Persistence Baseline in terms of accuracy and F1-score but performed less effectively than Logistic Regression.

These results demonstrate that machine learning can improve 30-minute-ahead occupancy prediction compared with a simple persistence-based approach for the selected room.

### 9.2 Key Challenges

Despite achieving the best overall performance, Logistic Regression produced a relatively high false positive rate of 60.62%.

It also overestimated the overall occupancy rate by approximately 12.18 percentage points.

These findings indicate that improved classification performance does not necessarily guarantee accurate estimation of occupancy proportions or optimal performance for building energy management.

Monthly evaluation further revealed variations in predictive performance, with Logistic Regression accuracy ranging from 63.70% to 78.02%.

Therefore, the reliability of the model may depend on changing occupancy patterns and evaluation periods.

### 9.3 Overall Conclusion

The project successfully developed and evaluated a reproducible workflow for 30-minute-ahead room occupancy prediction using historical occupancy sensor data.

Among the evaluated approaches, Logistic Regression provided the strongest overall performance on the held-out chronological test dataset.

However, the findings are limited to one selected room and a retrospective evaluation setting.

Future work should investigate improved classification thresholds, additional environmental features, multiple-room evaluation, and real-world deployment.

Overall, this study demonstrates the potential of relatively simple machine learning methods for short-term occupancy prediction while highlighting the importance of error analysis, temporal evaluation, and application-specific model selection.



## 10. Final Output Verification

This section verifies that the final prediction data, model performance summaries, classification error analysis, occupancy prediction bias results, and monthly performance summaries have been successfully saved.

These files provide supporting evidence for the final project report and help ensure that the experimental results are reproducible.


In [6]:

# Verify final project output files
expected_files = [
    "final_model_predictions.csv",
    "final_performance_summary.csv",
    "final_error_summary.csv",
    "final_occupancy_bias_summary.csv",
    "final_monthly_performance_summary.csv",
    "model_performance_comparison.png",
    "confusion_matrix_comparison.png",
    "classification_error_comparison.png",
    "occupancy_prediction_bias.png",
    "monthly_model_performance.png",
    "occupancy_rate_vs_f1.png",
    "monthly_prediction_bias.png"
]

missing_files = []

print("Notebook 05 - Final Output Verification\n")

for filename in expected_files:
    file_path = results_dir / filename

    if file_path.is_file():
        print(f"PASS: {filename}")
    else:
        print(f"MISSING: {filename}")
        missing_files.append(filename)

print(
    f"\nFiles verified: "
    f"{len(expected_files) - len(missing_files)}/{len(expected_files)}"
)

if not missing_files:
    print("All expected project output files are available.")
else:
    print("Some project output files are missing.")


Notebook 05 - Final Output Verification

PASS: final_model_predictions.csv
PASS: final_performance_summary.csv
PASS: final_error_summary.csv
PASS: final_occupancy_bias_summary.csv
PASS: final_monthly_performance_summary.csv
PASS: model_performance_comparison.png
PASS: confusion_matrix_comparison.png
PASS: classification_error_comparison.png
PASS: occupancy_prediction_bias.png
PASS: monthly_model_performance.png
PASS: occupancy_rate_vs_f1.png
PASS: monthly_prediction_bias.png

Files verified: 12/12
All expected project output files are available.
